# Brightfield Overview — Tiled Scan
Acquire a tiled widefield overview with the PCO camera and display as a mosaic.

## 1. Connect

In [ ]:
import sys, importlib
import numpy as np
import matplotlib.pyplot as plt
import tifffile, re
from pathlib import Path
from pycromanager import Core, Studio

sys.path.insert(0, r'C:\Users\aifadmin\scapecontrol\repo')
import opm_acquisition; importlib.reload(opm_acquisition)

core = Core()
studio = Studio()
print('Connected — camera:', core.get_camera_device())

## 2. Switch to widefield and preview

In [ ]:
from opm_acquisition import switch_to_widefield

LED_INTENSITY_PCT = 20
EXPOSURE_MS       = 20.0

switch_to_widefield(core)
core.set_property('LED:L:37:4', 'LED Intensity(%)', str(LED_INTENSITY_PCT))
core.set_exposure(EXPOSURE_MS)

core.snap_image()
tagged = core.get_tagged_image()
img = np.reshape(tagged.pix, [tagged.tags['Height'], tagged.tags['Width']]).astype(np.float32)

print(f'Camera: {core.get_camera_device()}  |  Shutter: {core.get_shutter_device()}')
print(f'Shape: {img.shape}  |  Min: {img.min():.0f}  Max: {img.max():.0f}  Mean: {img.mean():.0f}')

p1, p99 = np.percentile(img, (1, 99))
img_display = np.clip((img - p1) / (p99 - p1 + 1e-9), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_display, cmap='gray')
axes[0].set_title(f'Preview — LED {LED_INTENSITY_PCT}%  exp {EXPOSURE_MS} ms')
axes[0].axis('off')
axes[1].hist(img.ravel(), bins=256, color='gray')
axes[1].set_title('Pixel intensity histogram')
axes[1].set_xlabel('Intensity (raw)')
plt.tight_layout()
plt.show()

## 3. Grid settings — edit here

In [ ]:
# --- Camera (do not change unless recalibrated) ---
PIXEL_SIZE_UM = 0.36
CAMERA_PX     = 1024
FOV_UM        = CAMERA_PX * PIXEL_SIZE_UM   # ~369 µm

# --- Scan range (edit here) ---
# ibidi µ-Slide 8 Well: inner well dimensions ~6.9 × 6.9 mm
RANGE_X_UM = 6900   # total scan width  in µm
RANGE_Y_UM = 6900   # total scan height in µm
OVERLAP    = 0.10   # fractional tile overlap (0.10 = 10%)

# --- Derived ---
step_um = FOV_UM * (1 - OVERLAP)
half_x  = RANGE_X_UM / 2
half_y  = RANGE_Y_UM / 2

CUSTOM_GRID = dict(
    x_start = -half_x,
    x_end   =  half_x,
    y_start = -half_y,
    y_end   =  half_y,
    step_um =  step_um,
)

n_cols = round(RANGE_X_UM / step_um) + 1
n_rows = round(RANGE_Y_UM / step_um) + 1
print(f'FOV       : {FOV_UM:.0f} µm  ({PIXEL_SIZE_UM} µm/px)')
print(f'Step      : {step_um:.0f} µm  ({OVERLAP*100:.0f}% overlap)')
print(f'Grid      : {n_cols} cols × {n_rows} rows = {n_cols * n_rows} tiles')
print(f'Range     : {RANGE_X_UM:.0f} × {RANGE_Y_UM:.0f} µm  (~{RANGE_X_UM/1000:.1f} × {RANGE_Y_UM/1000:.1f} mm)')

## 4. Preview tile grid

In [ ]:
importlib.reload(opm_acquisition)
from opm_acquisition import _snake_grid

x_cur, y_cur = core.get_x_position(), core.get_y_position()
print(f'Current stage: x={x_cur:.1f}  y={y_cur:.1f} um')

events = _snake_grid(CUSTOM_GRID, center_x=x_cur, center_y=y_cur)
xs = [e['x'] for e in events]
ys = [e['y'] for e in events]

plt.figure(figsize=(7, 7))
plt.plot(xs, ys, 'o-', markersize=5, label='tiles')
plt.plot(x_cur, y_cur, 'r*', markersize=14, label='current pos (center)')
plt.xlabel('X (um)'); plt.ylabel('Y (um)')
plt.title(f'Tile grid: {len(events)} positions  step={CUSTOM_GRID["step_um"]:.0f} um')
plt.legend(); plt.axis('equal'); plt.grid(True)
plt.show()

## 5. Acquire

In [ ]:
importlib.reload(opm_acquisition)
from opm_acquisition import acquire_brightfield_overview

OUTPUT_DIR = Path(r'C:\Users\aifadmin\Desktop\TEST')

tiles = acquire_brightfield_overview(core, OUTPUT_DIR, grid=CUSTOM_GRID)
print(f'Done — {len(tiles)} tiles saved to {OUTPUT_DIR / "brightfield_overview"}')

## 6. Display mosaic

In [ ]:
OVERVIEW_DIR = Path(r'C:\Users\aifadmin\Desktop\TEST\brightfield_overview')

tile_files = sorted(OVERVIEW_DIR.glob('tile_r*_c*.tif'))
print(f'Found {len(tile_files)} tiles')

if not tile_files:
    raise RuntimeError('No tiles found — run step 5 first.')

def parse_rc(path):
    m = re.search(r'tile_r(\d+)_c(\d+)', path.name)
    return int(m.group(1)), int(m.group(2))

coords  = [parse_rc(f) for f in tile_files]
rows    = sorted(set(r for r, c in coords))
cols    = sorted(set(c for r, c in coords))
max_row = max(rows)

tile_map = {parse_rc(f): tifffile.imread(str(f)).astype(np.float32) for f in tile_files}
h, w    = next(iter(tile_map.values())).shape
mosaic  = np.zeros((len(rows) * h, len(cols) * w), dtype=np.float32)

for r in rows:
    for c in cols:
        if (r, c) in tile_map:
            img = tile_map[(r, c)]
            p1, p99 = np.percentile(img, (1, 99))
            img = np.clip((img - p1) / (p99 - p1 + 1e-9), 0, 1)
            mr = (max_row - r) * h   # higher Y row → top (+Y = up in world)
            mc = c * w               # col 0 → left
            mosaic[mr:mr + h, mc:mc + w] = img

fig, ax = plt.subplots(figsize=(14, 14))
ax.imshow(mosaic, cmap='gray', origin='upper')
ax.set_title(f'Brightfield overview — {len(rows)}×{len(cols)} tiles')
ax.axis('off')
plt.tight_layout()

mosaic_path = OVERVIEW_DIR / 'mosaic.tif'
tifffile.imwrite(str(mosaic_path), (mosaic * 65535).astype(np.uint16))
print(f'Mosaic saved: {mosaic_path}')

plt.show()
print(f'Mosaic: {mosaic.shape[1]} × {mosaic.shape[0]} px')